# Test Set

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
import re
import string

In [10]:
file_path = "/content/drive/MyDrive/exemption_test_set.xlsx"
df = pd.read_excel(file_path)

# Exemption Request Classification

## Keyword-based filtering

In [8]:
exemption_keywords = [ "derogation", "exemption", "exception", "temporary exception",
    "request for exemption", "temporary exemption", "time-limited derogation",
    "unlimited derogation", "unlimited exception" , "unlimited exemption", "necessary use", "essential function",
    "no viable alternative", "not replaceable", "unavailable substitute",
    "cannot be substituted", "substitution not feasible", "derogation granted",
    "critical use", "technological alternatives not available", "can not be substituted",
    "exempted application", "application should be exempted", "can not be replaced","cannot be replaced", "not provide alternatives", "no alternative", "FCJ",
]

In [9]:
def preprocess(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [10]:
def exemption_scraping(comment):
    processed_comment = preprocess(comment)
    for keyword in exemption_keywords:
        pattern = rf'{re.escape(keyword)}' if ' ' in keyword else rf'\b{re.escape(keyword)}\b'
        if re.search(pattern, processed_comment):
            return 1
    return 0

In [11]:
df["Prediction_1"] = df["Comment"].apply(exemption_scraping)

df.to_excel(file_path, index=False)
print(" Exemption predictions added and saved to:")
print(file_path)


 Exemption predictions added and saved to:
/content/drive/MyDrive/exemption_test_set.xlsx


## Few-shot Classification

In [4]:
import os
import openai
from tqdm import tqdm
import time

In [37]:
# SECRET
API_ENDPOINT = "..."
API_KEY = "..."

In [38]:
client = openai.OpenAI(
    api_key= API_KEY,
    base_url= API_ENDPOINT)

In [39]:
input_path = file_path
output_path = file_path
checkpoint_interval = 5

In [40]:
df = pd.read_excel(output_path if os.path.exists(output_path) else input_path)
print("Resuming from previous checkpoint" if os.path.exists(output_path) else "Starting from input")


current_output_col = "Prediction_2"

if current_output_col not in df.columns:
    df[current_output_col] = ""

Resuming from previous checkpoint


In [41]:
comment_1 = """Avery Dennison Corporation is a global materials science and digital identification solutions company that provides branding and information labeling solutions, including pressure-sensitive materials, radio-frequency identification (RFID) inlays and tags, and a variety of converted products and solutions. The company designs and manufactures a wide range of labeling and functional materials that enhance branded packaging, carry or display information that connects the physical and the digital, and improve customers’ product performance. The company serves an array of industries worldwide, including home and personal care, apparel, e-commerce, logistics, food and grocery, pharmaceuticals and automotive.  PFAS substances and materials are currently being phased out of the Avery Dennison portfolio,  which is consistent with the implementation of the PFAS restriction process.  We have, however, identified two uses where technological alternatives are currently not available, hence the request for a time limited derogation to engineer a substitute and phase out. For both these applications the components containing PFAS are purchased externally.  Use sector: Transport  Sub-use: Use of PFASs in applications affecting the proper functioning related to the safety of vehicles, and affecting the safety of operators, passengers or goods, to the extent not addressed under other parts of this proposed restriction. 9034 Information submitted confidentially 9034    The two missing uses identified are listed below and fall within the use sector "Transport". An assessment of alternatives and substitution timelines are outlined in detail in the confidential section of the submission.    1)Protective Overlay Films for Traffic Signs and Public Transport vehicles - These products are essential for the protection and durability of road signs and public transport vehicle signs which must retain their visibility in low-light conditions and withstand the long-term effects of weather and wear. Stringent regulatory standards apply to this use.  2)Release liners for pressure-sensitive silicone-based adhesives for Airbag labels (Sub-use 1) and Brake shims (Sub-Use 2). Avery Dennison uses a fluorosilicone release liner to protect the pressure-sensitive adhesive before its application, protecting it during shipment, storage, and converting. Sub-uses are highly regulated."""
comment_2 = """Water and oil repellent sprays used for shoes and sandals made of leather or canvas must have not only water repellency but also oil repellency from the standpoint of stain prevention. Oily substances such as cooking sauces and salad dressings soak into leather and fibers, causing stains. If leather products or silk Japanese clothes get dirty, cleaning is difficult and time-consuming. In some cases, these stains cannot be removed even by cleaning, and the product cannot be used. In order to avoid such troubles, it is essential to coat the surface with an oil-repellent fluorine-based substance.  Various resins such as acrylic and urethane resins, as well as silicone-based compounds, can be considered as substitutes for PFAS in this application, but compounds other than fluorine-based compounds do not have oil repellency and cannot prevent stains. In addition, it cannot be used as a substitute for this application because of its poor water repellency. 4121 Leather and textile protection (oil  and water repellent) 4121 There are no PFAS emissions during manufacturing and use. At the end of its useful life, the entire amount is disposed of, but almost all of it is incinerated and decomposed into HF, so no PFAS is released into the environment. 4121 At the end of its useful life, the entire amount is disposed of, but almost all of it is incinerated and decomposed into HF, so no PFAS is released into the environment. 4121  If we limit it to leather and silk products, we estimate that it is 100 to 500 tons each year. 4121    There is no academic data that it is a precursor of these substances. """
comment_3 = """We apply for exemption of following PFAS substances used in plastic materials and/or lubricants of our products POLYTETRAFLUOROETHYLENE (CAS No: 9002-84-0) TETRAFLUOROETHYLENE-HEXAFLUOROPROPYLENE COPOLYMER (CAS No: 25067-11-2) TETRAFLUOROETHYLENE (CAS No: 116-14-3) ETHENE,1,1,2,2-TETRAFLUORO-, OXIDIZED, POLYMD. (CAS No: 69991-61-3) POLY(PERFLUOROPROPYLENE OXIDE-CO-PERFLUOROFORMALDEHYDE) (CAS No: 69991-67-9) 1,1,2,3,3,3-HEXAFLUORO-1-PROPENE, OXIDIZED, POLYERMIZED (CAS No: 161075-14-5) FLUORINERT (CAS No: 86508-42-1) PERFLUOROALKYLETHER (CAS No: 60164-51-4) PERFLUOROOCTANE SULFONATE (PFOS) (CAS No: 1763-23-1) 7726 Electronics and Semiconductors-Electronic component 7726 These PFAS substances are only present in the plastic and/or lubricants of our electronic connector parts. We don't do material recycling and the end of life stage will depend on the end customer. 7726 These PFAS substances are only present in the plastic and/or lubricants of our electronic connector parts. We don't do material recycling and the end of life stage will depend on the end customer 7726 These PFAS substances are only present in the plastic and/or lubricants of our electronic connector parts. We don't do material recycling and the end of life stage will depend on the end customer. 7726 We have a material replacement plan in place and will use material containing PFAS substances 7726 We have a material replacement plan in place and will use material containing PFAS substances 7726 We have a material replacement plan in place and will use material containing PFAS substances 7726 We have a material replacement plan in place and will use material containing PFAS substances."""


comment_4 = """The proposal is good as it is and must not be watered down. I am willing to sacrifice certain characteristics of products, such as heavier phones or less waterproof rain jackets, if it means we can stop using PFAS. There is more and more research showing how bad this is for all of us. We will be ashamed and regretful in the future for not stopping this when we could. Please listen to the public and not just the companies!"""
comment_5 = """REINTJES Benelux BV appreciates the opportunity to provide the following comments in response to the proposal of this PFAS restriction. REINTJES supports the EU's efforts to protect human health and environment. However, this proposal does not seem to adequately address the impact on a wide range of industries. For this reason, we comment on this proposal as follows. A blanket restriction of the entire PFAS substance group without a differentiated, substance- and application-specific risk assessment and solely due to the persistence of some PFAS is not appropriate. In order to ultimately achieve a sustainable overall balance of resource conservation and environmental impact,  a restriction is only justified in cases where the risks to humans and the environment cannot be controlled. It should be taken into account that the PFAS definition includes substances with different properties and that neither all PFAS are equally persistent. This puts almost risk-free chemicals on an equal footing with substances of very high concern with properties that require regulation. As part of a differentiated approach, it is urgent to ensure that only those substances whose use poses an unacceptable risk to the environment or human health are banned. Otherwise, there is a risk that chemicals that play a crucial role in innovative technologies will be driven out of the market. For example, manufacturers of hydraulic components such as pumps, motors, valves and cylinders as well as manufacturers of valves and compressors are affected. PFAS, mostly fluorinated polymers, are often used in seals, hoses, pipes, valves and coatings that we absolutely need for our products. While in some cases "only" the performance of some products would be massively affected, some products could no longer be manufactured, which would mean a very high impact on not only our company, our customers and our market but the entire shipping industry. 8735 1. Sectors and (sub-)uses: - Electronics and semiconductor (Annex E.2.11.) - Construction products (Annex E.2.13.) - Lub"""
comment_6 = """General comments: Please refer to the attached file in Section V. 7108 1. Sectors and (sub-)uses: Sectors :      Transport (Sub-)Uses : Use of PFASs in applications affecting the proper functioning related to the safety of vehicles, and                       affecting the safety  of  operators, passengers or  goods, to  the extent not addressed under other                      parts of this proposed restriction. 7108 2. Emissions in the end-of-life phase: Please refer to the attached file in Section V. 7108 7. Potential derogations marked for reconsideration: Please refer to the attached files in Section V. 7108 8. Other identified uses: Please refer to the attached files in Section V."""

In [42]:
def few_grouped(comment):
    return f"""
The task is to determine whether a public consultation comment explicitly or implicitly requests an exemption or derogation from the proposed PFAS restriction policy.

You are a language model trained to identify whether a stakeholder is requesting an exemption in a public consultation comment regarding the PFAS restriction proposal.

Below are examples of previously classified comments:

Example 1:
Comment 1: {comment_1}
Label 1: 1

Example 2:
Comment 2: {comment_2}
Label 2: 1

Example 3:
Comment 3: {comment_3}
Label 3: 1

Example 4:
Comment 4: {comment_4}
Label 4: 0

Example 5:
Comment 5: {comment_5}
Label 5: 0

Example 6:
Comment 6: {comment_6}
Label 6: 0

Taking the examples into consideration, determine whether the following comment requests any exemption or derogation from the PFAS restriction policy:

Comment:
\"{comment}\"

Respond with only one of the following digits:
- 1 → if an exemption is requested
- 0 → if no exemption is requested

Answer with only 1 or 0.
"""


In [43]:
def get_response(prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f" Error: {e}")
        return "error"

In [44]:
# MAIN LOOP
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Classifying comments"):
    comment = str(row["Comment"])

    # STANCE CLASSIFICATION
    if not row["Prediction_2"]:
        prediction = get_response(few_grouped(comment))
        df.at[idx, "Prediction_2"] = prediction
        print(f"[{idx}] Exemption → {prediction}")
        time.sleep(1)

    # CHECKPOINT SAVE
    if idx % checkpoint_interval == 0:
        df.to_excel(output_path, index=False)
        print(f"Checkpoint saved at row {idx}")

# FINAL SAVE
df.to_excel(output_path, index=False)
print(" Classification completed and saved.")

Classifying comments:   0%|          | 0/244 [00:00<?, ?it/s]

[0] Exemption → 0


Classifying comments:   0%|          | 1/244 [00:01<07:06,  1.76s/it]

Checkpoint saved at row 0
[1] Exemption → 1


Classifying comments:   1%|          | 2/244 [00:03<06:04,  1.51s/it]

...

[242] Exemption → 1


Classifying comments: 100%|█████████▉| 243/244 [05:40<00:01,  1.37s/it]

[243] Exemption → 1


Classifying comments: 100%|██████████| 244/244 [05:41<00:00,  1.40s/it]

 Classification completed and saved.


# Evaluation

## Accuracy, Macro Precision, Macro Recall and Macro F1-score

In [11]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [12]:
results_path = "/content/drive/MyDrive/exemption_test_results.xlsx"


y_true = df["Exemption"]


model_outputs = [
    "Prediction_1",  "Prediction_2"]


if os.path.exists(results_path):
    results_df = pd.read_excel(results_path, index_col=0)
else:
    results_df = pd.DataFrame()


for model_name in model_outputs:
    if model_name not in df.columns:
        print(f" Skipping {model_name} (not found in dataset)")
        continue

    y_pred = df[model_name]


    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    _, _, f1_class, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=[0, 1])



    metrics = pd.Series({
    "Accuracy": accuracy,
    "Macro Precision": precision,
    "Macro Recall": recall,
    "Macro F1-score": f1_macro,
    "F1 Class 0": f1_class[0],
    "F1 Class 1": f1_class[1],}, name=f"model_{model_name.split('_')[1]}")



    results_df[metrics.name] = metrics


results_df.to_excel(results_path)
print(f" All model results saved to {results_path}")

 All model results saved to /content/drive/MyDrive/exemption_test_results.xlsx
